# 07 决策树 Decision Tree

依赖安装说明：`pip install numpy matplotlib scikit-learn`

决策树通过一连串“如果某个特征小于某个阈值，就往左，否则往右”的规则做预测。它可解释性强，也容易过拟合。


## 1. 数学逻辑

分类树常用 Gini impurity 衡量一个节点有多混杂：

$$Gini = 1 - \sum_k p_k^2$$

如果一个节点里全是同一类，Gini 为 0。每次分裂时，决策树寻找能让子节点更纯的特征和阈值：

$$Gain = Gini(parent) - \frac{n_L}{n}Gini(left) - \frac{n_R}{n}Gini(right)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y = make_moons(n_samples=240, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


In [ ]:
# 从零看一个节点如何选择最佳切分

def gini(labels):
    _, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    return 1 - np.sum(probs ** 2)

def best_stump_split(X, y):
    base = gini(y)
    best = {'gain': -1}
    for feature in range(X.shape[1]):
        for threshold in np.unique(X[:, feature]):
            left = X[:, feature] <= threshold
            if left.sum() == 0 or left.sum() == len(y):
                continue
            gain = base - left.mean() * gini(y[left]) - (~left).mean() * gini(y[~left])
            if gain > best['gain']:
                best = {'feature': feature, 'threshold': threshold, 'gain': gain}
    return best

print('根节点最佳一刀:', best_stump_split(X_train, y_train))


In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('accuracy:', round(accuracy_score(y_test, pred), 3))

plt.figure(figsize=(10, 5))
plot_tree(model, feature_names=['x1', 'x2'], class_names=['0', '1'], filled=True, rounded=True)
plt.title('深度为 3 的决策树')
plt.show()

xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 180),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 180))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = model.predict(grid).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap='coolwarm', edgecolor='k', s=25)
plt.title('决策树的阶梯状决策边界')
plt.show()


## 2. 常见误区

- 不限制深度的树很容易把训练集记住。
- 单棵树对数据扰动敏感，小变化可能导致结构变化很大。
- 特征重要性不一定等于因果重要性。

## 3. 小实验

- 改 `max_depth`，观察边界从简单到复杂。
- 改 `min_samples_leaf`，看过拟合是否缓解。
- 对比随机森林，观察稳定性提升。
